In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import re
import requests
from pathlib import Path
from wordcloud import WordCloud

In [ ]:
faculty = pd.read_csv("../data/faculty_list_with_fields.csv")
faculty["field"].value_counts()

In [ ]:
print(faculty[(faculty["field"] == "Unknown") & (faculty["subfield"] != "Unknown")].to_string())

In [ ]:
print(faculty[(faculty["title"].str.contains("Professor")) & (faculty["field"] == "Unknown") & (faculty["subfield"] == "Unknown")].to_string())

In [ ]:
# Only 1 subfield, might be incorrect.
print(faculty[(faculty["field"].notna()) & (faculty["field"] != "Unknown") & (faculty["subfield"].str.split(",").str.len() == 1)].to_string())

In [ ]:
print(faculty[faculty["field"] == "Statistics"].to_string())

In [ ]:
print(faculty[faculty["field"] == "Invalid"].to_string())

In [ ]:
print(faculty[(faculty["field"] == "Computer Science") & (faculty["subfield"].str.split(",").str.len() > 1)].to_string())

In [ ]:
# Check inconsistency
for _, row in faculty[faculty["url"].notna()].iterrows():
    name = row["name"]
    college = row["college"]
    field = row["field"]
    text = Path(f"../data/faculty_websites_cleaned/{college}/{name}.txt").read_text(encoding="utf8").lower()
    if "professor of computer" in text and field != "Computer Science":
        print(f"{name} - {college}: Text contains Professor of Computer but field is {field}")
        continue
    elif "professor of mathematics" in text and "professor of mathematics and computer" not in text and field != "Mathematics":
        print(f"{name} - {college}: Text contains Professor of Mathematics but field is {field}")
        continue
    cs_cnt = len(re.findall(r"comput", text))
    math_cnt = len(re.findall(r"math", text))
    stat_cnt = len(re.findall(r"statistic", text))
    non_cs_cnt = math_cnt + stat_cnt
    max_cnt = max(non_cs_cnt, cs_cnt)
    if max_cnt == 0:
        if field != "Unknown":
            print(f"{name} - {college}: Has 0 mentions of any field but field is {field}")
        else:
            continue
    elif (cs_cnt > non_cs_cnt and field != "Computer Science") or (cs_cnt < non_cs_cnt and field == "Computer Science"):
        print(f"{name} - {college}: Has {non_cs_cnt} mentions of 'math' and 'statistic', {cs_cnt} of 'comput', but field is {field}")
    elif cs_cnt == non_cs_cnt:
        print(f"{name} - {college}: Has {non_cs_cnt} mentions of 'comput' and 'math' + 'statistic' but field is {field}")


In [ ]:
subfields = ",".join(faculty[(faculty["field"] == "Computer Science") & (faculty["subfield"] != "Unknown")]["subfield"])
subfields = re.sub(r'human-computer interaction', 'HCI', subfields, flags=re.IGNORECASE)
subfields = re.sub(r'natural language processing', 'NLP', subfields, flags=re.IGNORECASE)
subfields = re.sub(r'ad hoc network', 'adhoc network', subfields, flags=re.IGNORECASE)
subfields = re.sub(r'health.care', 'healthcare', subfields, flags=re.IGNORECASE)

wordcloud = WordCloud(width=1600, height=800, background_color='white', colormap="tab20_r").generate(subfields)
plt.figure(figsize=(20, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.savefig("../wordcloud.pdf")
plt.show()